## Cell 1: Setup Library
Cell ini berfungsi untuk memanggil semua library dan mengunci lokasi folder data_fe/featured agar file dipastikan keluar dari folder notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os

## Cell 2: Pengaturan Path, Load Data Clean & Referensi WHO
Cell ini memuat data hasil pembersihan awal dan file standar gizi WHO untuk seluruh rentang usia.

In [7]:
FEATURED_DIR = "../data/data_fe/featured"
os.makedirs(FEATURED_DIR, exist_ok=True)

print(f"✅ Folder tujuan siap di: {FEATURED_DIR}")

df = pd.read_csv("../data/processed/dataset_stunting_clean.csv")
who_ref = pd.read_csv("../data/processed/who_reference_master.csv")

print(f"📊 Jumlah data awal (0-60 bulan): {len(df):,} sampel")
print(f"📊 Jumlah data referensi WHO    : {len(who_ref):,} baris")

✅ Folder tujuan siap di: ../data/data_fe/featured
📊 Jumlah data awal (0-60 bulan): 40,066 sampel
📊 Jumlah data referensi WHO    : 468 baris


## Cell 3: Proses Pembuatan Fitur Cerdas & Flag Kelompok Usia
Di sini kita membuat fitur turunan secara global (0-60 bulan) dan menyisipkan flag is_balita agar model AI bisa membedakan anak di bawah dan di atas 2 tahun dengan cerdas.

In [15]:
df_fe = df.copy()

# 1. Standardisasi Nama Kolom ke Huruf Kecil
df_fe.columns = df_fe.columns.str.lower().str.strip()
who_ref.columns = who_ref.columns.str.lower().str.strip()

# 2. Pemetaan Variabel Usia (Menyelaraskan 'usia_bulan' dan 'month' menjadi 'usia')
df_fe['usia'] = df_fe['usia_bulan']
who_ref['usia'] = who_ref['month']

# Filter data reference hanya untuk indikator TB/U agar merge selisih tinggi badan akurat
who_tbu = who_ref[who_ref['indicator'] == 'TB/U'].copy()

# 3. Fitur Indikator Kelompok Usia (Kunci utama untuk melatih AI semua umur)
df_fe['is_balita'] = np.where(df_fe['usia'] > 24, 1, 0) # 0: 0-24 bln, 1: 25-60 bln
df_fe['potensi_growth_spurts'] = np.where((df_fe['usia'] >= 12) & (df_fe['usia'] <= 24), 1, 0)

# 4. Fitur Antropometri Dasar & Rasio
df_fe['bmi'] = df_fe['berat_badan'] / ((df_fe['tinggi_badan'] / 100) ** 2)
df_fe['rasio_bb_tb'] = df_fe['berat_badan'] / df_fe['tinggi_badan']

# 5. Sinkronisasi Teks Jenis Kelamin
df_fe['jenis_kelamin'] = df_fe['jenis_kelamin'].str.lower().str.strip()
who_tbu['jenis_kelamin'] = who_tbu['jenis_kelamin'].str.lower().str.strip()

# 6. Penggabungan Global dengan Referensi Standar WHO TB/U
df_master_featured = pd.merge(df_fe, who_tbu[['usia', 'jenis_kelamin', 'sd0']], on=['usia', 'jenis_kelamin'], how='left')

# 7. Perhitungan Jarak & Selisih Nilai Baku WHO (sd0 adalah nilai median WHO)
df_master_featured['selisih_tb_who'] = df_master_featured['tinggi_badan'] - df_master_featured['sd0']
df_master_featured['persen_median_who'] = (df_master_featured['tinggi_badan'] / df_master_featured['sd0']) * 100

# 8. Encoding Karakter Utama & Tambahan Fitur Kombinasi
df_master_featured['jenis_kelamin_enc'] = np.where(df_master_featured['jenis_kelamin'] == 'laki-laki', 1, 0)
df_master_featured['zscore_mean'] = df_master_featured[['bmi', 'rasio_bb_tb']].mean(axis=1)

print(f"✅ Pembuatan fitur selesai! Total kolom saat ini: {df_master_featured.shape[1]}")
df_master_featured[['usia', 'is_balita', 'bmi', 'selisih_tb_who']].head()

✅ Pembuatan fitur selesai! Total kolom saat ini: 21


,usia,is_balita,bmi,selisih_tb_who
0,54.0,1,13.885602,-8.7
1,44.0,1,14.177694,-9.0
2,57.0,1,14.879371,-11.3
3,26.0,1,17.625381,-9.8
4,59.0,1,15.201999,-10.9


## Cell 4: Pemisahan Data & Imputasi Nilai Kosong
Cell ini berfungsi untuk memastikan data anak usia 2-5 tahun yang mungkin memiliki nilai kosong bisa terisi dengan aman menggunakan nilai median.

In [16]:
# 1. Membuat target biner dari kolom 'status_tb_u' asli (Stunting = 1, Tidak Stunting = 0)
df_master_featured['target_stunting'] = np.where(df_master_featured['status_tb_u'].str.lower() == 'stunting', 1, 0)
TARGET_COL = 'target_stunting'

# 2. Singkirkan semua kolom teks/kategori agar tidak merusak Scaler AI
kolom_dibuang = [TARGET_COL, 'id_anak', 'jenis_kelamin', 'status_bb_u', 'status_tb_u', 'status_bb_tb']
X = df_master_featured.drop(columns=kolom_dibuang, errors='ignore')
y = df_master_featured[TARGET_COL]

# 3. Ambil kolom yang bertipe angka saja
num_cols = X.select_dtypes(include=[np.number]).columns
X_numeric = X[num_cols]

# 4. Imputasi menggunakan strategi Median (Mengisi kolom sd0 yang kosong pada sebagian data balita)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X_numeric), columns=num_cols)

print(f"✅ Imputasi data selesai dengan aman!")
print(f"📊 Jumlah total data global siap scale: {X_imputed.shape[0]:,} sampel")
print(f"🤖 Jumlah fitur numerik untuk AI      : {X_imputed.shape[1]} fitur")

✅ Imputasi data selesai dengan aman!
📊 Jumlah total data global siap scale: 40,893 sampel
🤖 Jumlah fitur numerik untuk AI      : 16 fitur


## Cell 5: Global Scaling (StandardScaler) 0-60 Bulan
Cell ini bertugas menyamakan skala semua fitur angka agar model AI tidak bias dan bisa belajar dengan performa optimal untuk seluruh rentang usia.

In [17]:
# Inisialisasi StandardScaler
scaler = StandardScaler()

# Melakukan scaling pada data numerik yang sudah bersih dari missing value
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=num_cols)

# Satukan kembali fitur hasil scaling dengan kolom label target utamanya
dataset_ai_scaled_v2 = X_scaled.copy()
dataset_ai_scaled_v2[TARGET_COL] = y.reset_index(drop=True)

print("✅ Standarisasi data global (0-60 bulan) selesai dilakukan!")
print(f"📊 Dimensi dataset hasil scaling: {dataset_ai_scaled_v2.shape[0]} baris x {dataset_ai_scaled_v2.shape[1]} kolom")
dataset_ai_scaled_v2.head()

✅ Standarisasi data global (0-60 bulan) selesai dilakukan!
📊 Dimensi dataset hasil scaling: 40893 baris x 17 kolom


,usia_bulan,berat_badan,tinggi_badan,zscore_bb_u,zscore_tb_u,zscore_bb_tb,usia,is_balita,potensi_growth_spurts,bmi,rasio_bb_tb,sd0,selisih_tb_who,persen_median_who,jenis_kelamin_enc,zscore_mean,target_stunting
0,1.346820,0.053704,0.984040,-0.022701,-0.314561,-0.015037,1.346820,0.72167,-0.555835,-0.020832,-0.001348,1.154950,-0.691742,-0.459168,-1.101446,-0.020738,1
1,0.710405,0.012618,0.515045,-0.022246,-0.407236,-0.014095,0.710405,0.72167,-0.555835,-0.018474,-0.009589,0.741300,-0.761408,-0.616690,0.907897,-0.018432,1
2,1.537745,0.081094,0.941404,-0.021792,-0.710535,-0.008710,1.537745,0.72167,-0.555835,-0.012810,0.013546,1.322001,-1.295510,-0.950468,0.907897,-0.012683,1
3,-0.435143,-0.021620,-0.593487,-0.004753,-1.157060,0.006905,-0.435143,0.72167,-0.555835,0.009359,0.005072,-0.229186,-0.947182,-1.082406,0.907897,0.009338,1
4,1.665028,0.101637,1.026676,-0.016339,-0.634710,-0.004672,1.665028,0.72167,-0.555835,-0.010205,0.021288,1.369730,-1.202622,-0.857376,-1.101446,-0.010053,1


## Cell 6: Ekspor File Akhir ke Folder data_fe/featured
Cell penutup ini menggunakan path relatif ../data/... sehingga filenya dijamin otomatis melompat keluar dari folder notebooks/ tempat script berada, lalu mendarat dengan aman ke folder data proyekmu.

In [18]:
# Tentukan path file lengkap secara dinamis ke folder data_fe/featured
path_master_featured = os.path.join(FEATURED_DIR, "dataset_master_featured.csv")
path_ai_scaled = os.path.join(FEATURED_DIR, "dataset_ai_scaled_v2.csv")

# Ekspor DataFrame ke format CSV
df_master_featured.to_csv(path_master_featured, index=False)
dataset_ai_scaled_v2.to_csv(path_ai_scaled, index=False)

print("==================== EKSPOR FINAL GABUNGAN BERHASIL ====================")
print(f"📁 Lokasi Folder Simpan : {FEATURED_DIR}")
print(f"✅ File 1 Sukses Terbuat : dataset_master_featured.csv (Data Lengkap Unscaled)")
print(f"✅ File 2 Sukses Terbuat : dataset_ai_scaled_v2.csv    (Versi Latihan AI Global)")
print(f"📊 Total Data Terproses : {len(dataset_ai_scaled_v2):,} sampel balita (0-60 Bulan)")
print("=======================================================================")

==================== EKSPOR FINAL GABUNGAN BERHASIL ====================
📁 Lokasi Folder Simpan : ../data/data_fe/featured
✅ File 1 Sukses Terbuat : dataset_master_featured.csv (Data Lengkap Unscaled)
✅ File 2 Sukses Terbuat : dataset_ai_scaled_v2.csv    (Versi Latihan AI Global)
📊 Total Data Terproses : 40,893 sampel balita (0-60 Bulan)
